## Bước 1: Phân tích Tương quan (Correlation Analysis)
Trong bước này, chúng ta sẽ xem xét ma trận tương quan giữa các đặc trưng (features) và biến mục tiêu (target) `AP Ratio` (Năng suất).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Đọc dữ liệu
data_path = r'../Data Collection/Bangladesh_database_Final_Merged.csv'
df = pd.read_csv(data_path)

# Cột AP Ratio có một số giá trị lỗi '#DIV/0!' (chia cho 0), ta sẽ ép kiểu sang số và những lỗi này thành NaN
df['AP Ratio'] = pd.to_numeric(df['AP Ratio'], errors='coerce')

# Bỏ qua các cột không phải là dạng số để tính toán tương quan
numeric_df = df.select_dtypes(include=['number'])

# Tính ma trận tương quan (bỏ qua NaN)
corr = numeric_df.corr()

# Hiển thị biểu đồ Heatmap
plt.figure(figsize=(20, 16))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Matrix of Agricultural Data', fontsize=18)
plt.show()


In [ ]:
# Xem xét 10 đặc trưng có tương quan cao nhất với năng suất (AP Ratio)
print("Top đặc trưng có tương quan dương với năng suất:")
print(corr['AP Ratio'].sort_values(ascending=False).head(5))

print("\nTop đặc trưng có tương quan âm với năng suất:")
print(corr['AP Ratio'].sort_values(ascending=True).head(5))

# Kiểm tra hiện tượng Đa cộng tuyến (Multicollinearity) - tương quan nội bộ > 0.8
print("\nCác cặp đặc trưng có tương quan cao (Giá trị tuyệt đối > 0.8):")
highly_correlated = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.8:
            highly_correlated.append((corr.columns[i], corr.columns[j], corr.iloc[i, j]))

for pair in highly_correlated:
    print(f"{pair[0]} - {pair[1]}: {pair[2]:.2f}")


### 📊 Đánh giá Ma trận Tương quan (Correlation Evaluation)

**1. Tương quan với biến mục tiêu (`AP Ratio`)**
* Có thể thấy tất cả các đặc trưng đều có hệ số tương quan tuyến tính **rất thấp** đối với biến mục tiêu `AP Ratio` (dao động từ -0.11 đến 0.10). 
* Các đặc trưng có tương quan lớn nhất với biến mục tiêu chỉ là `Temp_Max` (~0.1), `FPAR` (-0.11), `NDVI_Season_Mean` (-0.10).
* **Kết luận:** Mối quan hệ giữa năng suất cây trồng (AP Ratio) và các yếu tố tự nhiên (đất đai, khí hậu, vệ tinh) là một **mối quan hệ phi tuyến tính (non-linear)** rất phức tạp. Điều này cho thấy các mô hình hồi quy tuyến tính (Linear Regression) sẽ không hiệu quả. Chúng ta bắt buộc phải sử dụng các mô hình học máy phi tuyến phức tạp như Random Forest, XGBoost, hoặc CatBoost.

**2. Hiện tượng Đa cộng tuyến (Multicollinearity)**
Từ phân tích các cặp có tương quan lớn hơn 0.8, chúng ta có thể kết luận rằng bộ dữ liệu đang bị đa cộng tuyến khá nghiêm trọng ở một số chiều dữ liệu:
* **Nhóm Thực vật (Vegetation Indices):** `FPAR` tương quan cực mạnh với `LAI` (0.85) và `NDVI_Season_Mean` (0.85). Chúng mang thông tin lặp lại về mật độ/độ xanh của cây.
* **Nhóm Khí hậu - Lượng mưa:** `Rain_Temp_Ratio` tương quan hoàn hảo với `Rainfall` (1.00). Do tỷ lệ này được chia từ lượng mưa, chúng mang thông tin y hệt nhau.
* **Nhóm Khí hậu - Nhiệt độ:** `Temp_Min` tương quan mạnh với `Temp_Mean` (0.87); `Temp_Max` tương quan mạnh với `Heat_Stress_Days` (0.89); và `LST_Kelvin` (Nhiệt độ bề mặt) tương quan rất mạnh với `Temp_Mean` (0.91).

**🛠️ Khuyến nghị cho Bước Tiếp theo (Preprocessing Recommendations):**
- **Xử lý giá trị Missing (`#DIV/0!`)**: Cột `AP Ratio` cần drop các dòng NaN vì đây là biến mục tiêu (target).
- **Loại bỏ đặc trưng thừa (Drop Features)**: Cần loại bỏ một trong các biến bị đa cộng tuyến để tránh gây nhiễu cho mô hình. 
  - Bỏ `Rain_Temp_Ratio` (giữ lại `Rainfall` vì tính mô tả cao hơn).
  - Bỏ `Temp_Mean` (chỉ giữ `Temp_Max` và `Temp_Min`).
  - Bỏ `LST_Kelvin` hoặc cân nhắc giữ lại và bỏ các Temp vì thông tin lặp.
  - Về nhóm viễn thám, xem xét chỉ giữ lại 2 chỉ số mạnh nhất (VD: `NDVI_Season_Mean` và `EVI`), có thể bỏ bớt `FPAR` hoặc `LAI` nếu mô hình bị Overfitting.
